In [ ]:
import pandas as pd
import plotly.express as px
import streamlit as st

from tratamento import (
    ANO_MAXIMO_VALIDO,
    COLUNAS_REGIOES,
    NOMES_REGIOES,
    carregar_dados_brutos,
    tratar_dados,
)

TEMPLATE = "plotly_white"
COR_PRINCIPAL = "#4F46E5"
CORES = px.colors.qualitative.Set2

METRICAS = {
    "Global": "vendas_global",
    "América do Norte": "vendas_na",
    "Europa": "vendas_eu",
    "Japão": "vendas_jp",
    "Outras regiões": "vendas_outras",
}
ROTULO_VENDAS = "Vendas (milhões de unidades)"

AGRUPAMENTOS = {"Sem agrupamento": None, "Gênero": "genero", "Fabricante": "fabricante"}


def _finalizar(fig, altura=430):

    fig.update_layout(template=TEMPLATE, height=altura, margin=dict(l=10, r=10, t=60, b=10))
    return fig


def _soma_por(df, chave, coluna):

    return df.groupby(chave, as_index=False)[coluna].sum().sort_values(coluna, ascending=False)


def grafico_vendas_por_genero(df, coluna="vendas_global", rotulo="Global"):
    dados = _soma_por(df, "genero", coluna)
    fig = px.bar(
        dados, x="genero", y=coluna, text_auto=".0f",
        title=f"Vendas totais por gênero - {rotulo}",
        labels={"genero": "Gênero", coluna: ROTULO_VENDAS},
        color_discrete_sequence=[COR_PRINCIPAL],
    )
    fig.update_traces(textposition="outside", cliponaxis=False)
    return _finalizar(fig)


def grafico_top_plataformas(df, coluna="vendas_global", rotulo="Global", top_n=10):
    dados = _soma_por(df, ["plataforma", "fabricante"], coluna).head(top_n)
    fig = px.bar(
        dados, x="plataforma", y=coluna, color="fabricante", text_auto=".0f",
        title=f"Top {top_n} plataformas por vendas - {rotulo}",
        labels={"plataforma": "Plataforma", coluna: ROTULO_VENDAS, "fabricante": "Fabricante"},
        color_discrete_sequence=CORES,
    )
    fig.update_xaxes(categoryorder="total descending")
    return _finalizar(fig)


def grafico_top_publishers(df, coluna="vendas_global", rotulo="Global", top_n=10):
    dados = _soma_por(df[df["publisher"] != "Desconhecido"], "publisher", coluna).head(top_n)
    fig = px.bar(
        dados.sort_values(coluna), x=coluna, y="publisher", orientation="h", text_auto=".0f",
        title=f"Top {top_n} publishers por vendas - {rotulo}",
        labels={"publisher": "Publisher", coluna: ROTULO_VENDAS},
        color_discrete_sequence=[COR_PRINCIPAL],
    )
    fig.update_traces(textposition="outside", cliponaxis=False)
    return _finalizar(fig)


def grafico_top_jogos(df, coluna="vendas_global", rotulo="Global", top_n=10):
    dados = df.nlargest(top_n, coluna)
    fig = px.bar(
        dados.sort_values(coluna), x=coluna, y="nome_plataforma", color="genero", orientation="h",
        title=f"Top {top_n} jogos mais vendidos - {rotulo}",
        labels={"nome_plataforma": "Jogo (plataforma)", coluna: ROTULO_VENDAS, "genero": "Gênero"},
        color_discrete_sequence=CORES,
    )
    return _finalizar(fig, altura=max(380, 38 * top_n + 120))


def grafico_vendas_por_ano(df, coluna="vendas_global", rotulo="Global", agrupar_por=None):
    chaves = ["ano"] + ([agrupar_por] if agrupar_por else [])
    dados = df.groupby(chaves, as_index=False)[coluna].sum()
    fig = px.line(
        dados, x="ano", y=coluna, color=agrupar_por, markers=True,
        title=f"Vendas por ano de lançamento - {rotulo}",
        labels={"ano": "Ano de lançamento", coluna: ROTULO_VENDAS,
                "genero": "Gênero", "fabricante": "Fabricante"},
        color_discrete_sequence=CORES if agrupar_por else [COR_PRINCIPAL],
    )
    return _finalizar(fig)


def grafico_participacao_regional_tempo(df):

    dados = df.groupby("ano", as_index=False)[COLUNAS_REGIOES].sum()
    longo = dados.melt(id_vars="ano", var_name="regiao", value_name="vendas")
    longo["regiao"] = longo["regiao"].map(NOMES_REGIOES)
    fig = px.area(
        longo, x="ano", y="vendas", color="regiao", groupnorm="percent",
        title="Participação de cada região nas vendas, por ano de lançamento",
        labels={"ano": "Ano de lançamento", "vendas": "% das vendas", "regiao": "Região"},
        color_discrete_sequence=CORES,
    )
    return _finalizar(fig)


def grafico_regioes_por_genero(df, percentual=False):
    soma = df.groupby("genero")[COLUNAS_REGIOES].sum()
    soma = soma.loc[soma.sum(axis=1).sort_values(ascending=False).index]
    longo = soma.reset_index().melt(id_vars="genero", var_name="regiao", value_name="vendas")
    longo["regiao"] = longo["regiao"].map(NOMES_REGIOES)
    fig = px.bar(
        longo, x="genero", y="vendas", color="regiao",
        barmode="relative" if percentual else "group",
        barnorm="percent" if percentual else None,
        title="Vendas por gênero e região" + (" (participação %)" if percentual else ""),
        labels={"genero": "Gênero", "vendas": "% das vendas do gênero" if percentual else ROTULO_VENDAS,
                "regiao": "Região"},
        color_discrete_sequence=CORES,
    )
    return _finalizar(fig)


def grafico_heatmap_fabricante_genero(df, coluna="vendas_global", rotulo="Global"):
    tabela = df.pivot_table(index="fabricante", columns="genero", values=coluna, aggfunc="sum", fill_value=0)
    fig = px.imshow(
        tabela, text_auto=".0f", aspect="auto", color_continuous_scale="Blues",
        title=f"Vendas por fabricante e gênero - {rotulo} (milhões)",
        labels={"x": "Gênero", "y": "Fabricante", "color": "Vendas (milhões)"},
    )
    return _finalizar(fig)


def grafico_dispersao_na_eu(df):

    dados = df[(df["vendas_na"] > 0) & (df["vendas_eu"] > 0)]
    fig = px.scatter(
        dados, x="vendas_na", y="vendas_eu", color="genero", opacity=0.55,
        log_x=True, log_y=True, hover_name="nome_plataforma",
        title="Vendas na América do Norte x Europa (cada ponto é um jogo, escala log)",
        labels={"vendas_na": "América do Norte (milhões, escala log)",
                "vendas_eu": "Europa (milhões, escala log)", "genero": "Gênero"},
        color_discrete_sequence=CORES,
    )
    return _finalizar(fig, altura=480)


def grafico_histograma_vendas(df, n_bins=40, limitar_p95=True):

    dados = df
    sufixo = ""
    if limitar_p95:
        corte = df["vendas_global"].quantile(0.95)
        dados = df[df["vendas_global"] <= corte]
        sufixo = f" (até o percentil 95: {corte:.2f} mi)"
    fig = px.histogram(
        dados, x="vendas_global", nbins=n_bins,
        title="Distribuição das vendas globais por jogo" + sufixo,
        labels={"vendas_global": "Vendas globais do jogo (milhões)"},
        color_discrete_sequence=[COR_PRINCIPAL],
    )
    fig.update_yaxes(title_text="Nº de jogos")
    return _finalizar(fig)


def grafico_boxplot_genero(df):
    ordem = df.groupby("genero")["vendas_global"].median().sort_values(ascending=False).index.tolist()
    fig = px.box(
        df, x="genero", y="vendas_global", color="genero", log_y=True,
        category_orders={"genero": ordem},
        title="Distribuição das vendas globais por gênero (escala log, ordenado pela mediana)",
        labels={"genero": "Gênero", "vendas_global": "Vendas globais do jogo (milhões, escala log)"},
        color_discrete_sequence=CORES,
    )
    fig.update_layout(showlegend=False)
    return _finalizar(fig, altura=480)


st.set_page_config(page_title="Vendas de Videogames", page_icon="🎮", layout="wide")


@st.cache_data
def carregar_dados():

    bruto = carregar_dados_brutos()
    return bruto, tratar_dados(bruto)


def mostrar_grafico(fig):

    try:
        st.plotly_chart(fig, width="stretch")
    except TypeError:
        st.plotly_chart(fig, use_container_width=True)


def mostrar_tabela(df):
    try:
        st.dataframe(df, width="stretch")
    except TypeError:
        st.dataframe(df, use_container_width=True)


def milhoes(valor):

    return f"{valor:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".") + " mi"


def gerar_conclusoes(df):

    frases = []
    total = df["vendas_global"].sum()

    por_genero = df.groupby("genero")["vendas_global"].sum().sort_values(ascending=False)
    frases.append(f"**Gêneros:** {por_genero.index[0]} lidera com {milhoes(por_genero.iloc[0])} "
                  f"({por_genero.iloc[0] / total * 100:.1f}% das vendas), seguido de {por_genero.index[1]}."
                  if len(por_genero) > 1 else
                  f"**Gêneros:** {por_genero.index[0]} lidera com {milhoes(por_genero.iloc[0])}.")

    por_plataforma = df.groupby("plataforma")["vendas_global"].sum().sort_values(ascending=False)
    frases.append(f"**Plataformas:** {por_plataforma.index[0]} é a mais vendida ({milhoes(por_plataforma.iloc[0])}).")

    top = df.loc[df["vendas_global"].idxmax()]
    frases.append(f"**Jogo mais vendido:** {top['nome']} ({top['plataforma']}, {top['ano']}) com {milhoes(top['vendas_global'])}.")

    por_ano = df.groupby("ano")["vendas_global"].sum()
    frases.append(f"**Tempo:** o ano de lançamento com mais vendas foi {por_ano.idxmax()} ({milhoes(por_ano.max())}).")

    partes = {nome: df[col].sum() / total * 100 for col, nome in NOMES_REGIOES.items()}
    texto = ", ".join(f"{nome} {valor:.1f}%" for nome, valor in partes.items())
    frases.append(f"**Regiões (participação nas vendas globais):** {texto}.")

    for col, nome in [("vendas_na", "América do Norte"), ("vendas_eu", "Europa"), ("vendas_jp", "Japão")]:
        lider = df.groupby("genero")[col].sum().sort_values(ascending=False)
        frases.append(f"**Gênero líder em {nome}:** {lider.index[0]} ({milhoes(lider.iloc[0])}).")

    if len(df) > 2:
        corr = df["vendas_na"].corr(df["vendas_eu"])
        frases.append(f"**Correlação entre vendas na América do Norte e na Europa:** {corr:.2f}.")
    return frases


bruto, dados = carregar_dados()

st.sidebar.title("🔎 Filtros")

ano_min, ano_max = int(dados["ano"].min()), int(dados["ano"].max())
periodo = st.sidebar.slider("Período (ano de lançamento)", ano_min, ano_max, (ano_min, ano_max))

opcoes_genero = sorted(dados["genero"].unique())
generos = st.sidebar.multiselect("Gêneros", opcoes_genero, default=opcoes_genero)

opcoes_fabricante = sorted(dados["fabricante"].unique())
fabricantes = st.sidebar.multiselect("Fabricante do console", opcoes_fabricante, default=opcoes_fabricante)

filtrado = dados[
    dados["ano"].between(*periodo)
    & dados["genero"].isin(generos)
    & dados["fabricante"].isin(fabricantes)
]
st.sidebar.caption(f"{len(filtrado):,} de {len(dados):,} jogos selecionados".replace(",", "."))

st.title("🎮 Vendas de Videogames pelo Mundo")
st.markdown(
    "A indústria de games movimenta bilhões e se comporta de forma diferente em cada mercado. "
    "Este painel explora **quais gêneros, plataformas e jogos mais venderam**, **como as vendas mudaram ao longo "
    "dos anos** e **quais as diferenças entre América do Norte, Europa e Japão**. "
    "Use os filtros da barra lateral e os controles de cada aba para explorar."
)

with st.expander("📚 Sobre o conjunto de dados"):
    st.markdown(
        f"""
- **Fonte:** [Video Game Sales (Kaggle)](https://www.kaggle.com/datasets/gregorut/videogamesales), com dados do VGChartz.
- **Tamanho original:** {bruto.shape[0]} jogos e {bruto.shape[1]} colunas; após o tratamento ficaram {len(dados)} jogos.
- **Variáveis:** nome, plataforma, ano de lançamento, gênero, publisher e vendas (em **milhões de unidades**)
  na América do Norte, Europa, Japão, outras regiões e no mundo.
- **Tratamento:** removidos jogos sem ano ou com ano acima de {ANO_MAXIMO_VALIDO} (dados quase inexistentes e um erro evidente);
  publishers ausentes viraram "Desconhecido"; criamos as colunas *fabricante* e *década* e traduzimos os gêneros.
- **Atenção:** os dados do VGChartz têm cobertura parcial nos anos mais recentes (2016 está incompleto).
        """
    )

if filtrado.empty:
    st.warning("Nenhum jogo corresponde aos filtros escolhidos. Ajuste os filtros na barra lateral.")
    st.stop()

total_global = filtrado["vendas_global"].sum()
genero_lider = filtrado.groupby("genero")["vendas_global"].sum().idxmax()
plataforma_lider = filtrado.groupby("plataforma")["vendas_global"].sum().idxmax()
jogo_top = filtrado.loc[filtrado["vendas_global"].idxmax()]

k1, k2, k3, k4, k5 = st.columns(5)
k1.metric("Jogos", f"{len(filtrado)}")
k2.metric("Vendas globais", milhoes(total_global))
k3.metric("Gênero líder", genero_lider)
k4.metric("Plataforma líder", plataforma_lider)
k5.metric("Jogo mais vendido", jogo_top["nome"])

st.divider()

aba1, aba2, aba3, aba4, aba5, aba6 = st.tabs([
    "🏆 Gêneros, plataformas e jogos", "📅 Evolução no tempo", "🌍 Regiões",
    "📦 Distribuição e relações", "📝 Conclusões", "🗂️ Dados",
])

with aba1:
    c_a, c_b = st.columns(2)
    rotulo_metrica = c_a.selectbox("Medida de vendas", list(METRICAS.keys()), key="metrica_aba1")
    top_n = c_b.slider("Quantidade de itens nos rankings", 3, 20, 10, key="top_n")
    coluna_metrica = METRICAS[rotulo_metrica]

    col1, col2 = st.columns(2)
    with col1:
        mostrar_grafico(grafico_vendas_por_genero(filtrado, coluna_metrica, rotulo_metrica))
    with col2:
        mostrar_grafico(grafico_top_plataformas(filtrado, coluna_metrica, rotulo_metrica, top_n))
    col3, col4 = st.columns(2)
    with col3:
        mostrar_grafico(grafico_top_jogos(filtrado, coluna_metrica, rotulo_metrica, top_n))
    with col4:
        mostrar_grafico(grafico_top_publishers(filtrado, coluna_metrica, rotulo_metrica, top_n))

with aba2:
    c_a, c_b = st.columns(2)
    rotulo_tempo = c_a.selectbox("Medida de vendas", list(METRICAS.keys()), key="metrica_aba2")
    rotulo_agrupar = c_b.selectbox("Separar a linha por", list(AGRUPAMENTOS.keys()))
    mostrar_grafico(grafico_vendas_por_ano(
        filtrado, METRICAS[rotulo_tempo], rotulo_tempo, AGRUPAMENTOS[rotulo_agrupar]))
    mostrar_grafico(grafico_participacao_regional_tempo(filtrado))

with aba3:
    modo = st.radio("Como comparar as regiões?", ["Valores absolutos", "Participação (%)"], horizontal=True)
    mostrar_grafico(grafico_regioes_por_genero(filtrado, percentual=(modo == "Participação (%)")))
    rotulo_heat = st.selectbox("Medida de vendas do mapa de calor", list(METRICAS.keys()), key="metrica_aba3")
    mostrar_grafico(grafico_heatmap_fabricante_genero(filtrado, METRICAS[rotulo_heat], rotulo_heat))

with aba4:
    c_a, c_b = st.columns(2)
    n_bins = c_a.slider("Intervalos do histograma", 10, 100, 40)
    limitar = c_b.checkbox("Ocultar jogos acima do percentil 95 (outliers)", value=True)
    col1, col2 = st.columns(2)
    with col1:
        mostrar_grafico(grafico_histograma_vendas(filtrado, n_bins, limitar))
    with col2:
        mostrar_grafico(grafico_boxplot_genero(filtrado))
    mostrar_grafico(grafico_dispersao_na_eu(filtrado))

with aba5:
    st.subheader("Principais resultados (atualizam com os filtros)")
    for frase in gerar_conclusoes(filtrado):
        st.markdown(f"- {frase}")
    st.caption("Vendas em milhões de unidades. Fonte: Kaggle / VGChartz.")

with aba6:
    st.subheader("Dados filtrados")
    mostrar_tabela(filtrado)
    st.download_button(
        "⬇️ Baixar dados filtrados (CSV)",
        data=filtrado.to_csv(index=False).encode("utf-8"),
        file_name="vgsales_filtrado.csv",
        mime="text/csv",
    )